## tl;dr

O AA2 materializa oito perguntas pré-registradas sem selecionar apenas resultados positivos. A trajetória de abandono de Nova Santa Rita em 2025 ficou dentro da faixa de municípios com contexto semelhante; a decomposição 2018–2025 indica que a pressão demográfica reduziria cerca de 41 matrículas, enquanto a relação territorial compensou com cerca de 58, encerrando com aumento observado de 17. A correspondência normativa entre oferta técnica local e ocupações é zero no município porque a matrícula EPT de 2025 é zero observado, embora a oferta acessível no Vale alcance 46,5% dos vínculos locais. As demais relações não atingiram o conjunto pré-registrado de efeito, incerteza, multiplicidade e robustez, ou ficaram insuficientes por cobertura.

## Context & Methods

Este é um caderno de auditoria do pacote AA2, não a fonte canônica dos estimadores. Ele relê os artefatos materializados por dois processos independentes e refaz controles de multiplicidade e fechamento.

### Key Assumptions

- o código IBGE permanece texto de sete dígitos;
- efeito e intervalo/limite precedem significância;
- associações ecológicas não são interpretadas como causalidade individual;
- resultados insuficientes e indisponíveis permanecem no pacote.

In [ ]:
from pathlib import Path
import json
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'package.json').exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / 'package.json').exists())
aa2_root = repo_root / '.tmp' / 'vocacoes-pne' / 'advanced-analytics-v1' / 'aa2'
assert aa2_root.is_dir(), aa2_root


## Data

Fontes diretas: `RESULTADOS_AA2.csv.gz`, `ROBUSTEZ_AA2.csv.gz`, `COMPARACOES_ESCOPO_AA2.csv.gz`, `CLAIMS_AA2.json`, `QA_SUMMARY_AA2.json` e `MANIFEST_AA2.json`. Os hashes dos insumos AA1, pré-registro, probe e ponte curso–CBO ficam no manifesto e no QA.

In [ ]:
results = pd.read_csv(aa2_root / 'RESULTADOS_AA2.csv.gz')
robustness = pd.read_csv(aa2_root / 'ROBUSTEZ_AA2.csv.gz')
scope = pd.read_csv(aa2_root / 'COMPARACOES_ESCOPO_AA2.csv.gz', dtype={'municipality_ibge_code': str})
claims = json.loads((aa2_root / 'CLAIMS_AA2.json').read_text(encoding='utf-8'))
qa = json.loads((aa2_root / 'QA_SUMMARY_AA2.json').read_text(encoding='utf-8'))
manifest = json.loads((aa2_root / 'MANIFEST_AA2.json').read_text(encoding='utf-8'))
assert qa['failedCount'] == 0
assert claims['questionCount'] == 8
assert manifest['independentMaterializationVerification']['equal'] is True
print({'results': len(results), 'robustness': len(robustness), 'scope': len(scope), 'qa_controls': qa['controlCount']})


## Results

A tabela abaixo mantém um estado terminal por pergunta. `NO_ROBUST_ASSOCIATION` não significa prova de ausência; indica apenas que o conjunto de critérios pré-registrados não foi satisfeito.

In [ ]:
terminal = (results[['question_id', 'terminal_state']]
            .drop_duplicates()
            .sort_values('question_id'))
assert terminal.groupby('question_id').size().eq(1).all()
print(terminal.to_string(index=False))


In [ ]:
# Recomputação independente do BH com slots inválidos preenchidos por p=1 apenas internamente.
for family, family_rows in results[results['multiplicity_family'].notna()].groupby('multiplicity_family'):
    ordered_rows = family_rows.sort_values('result_id').reset_index(drop=True)
    raw = [1.0 if pd.isna(value) else float(value) for value in ordered_rows['p_value_raw']]
    order = sorted(range(len(raw)), key=lambda index: (raw[index], ordered_rows.loc[index, 'result_id']))
    adjusted = [None] * len(raw)
    running = 1.0
    for position in range(len(raw) - 1, -1, -1):
        index = order[position]
        running = min(running, min(1.0, raw[index] * len(raw) / (position + 1)))
        adjusted[index] = running
    for index, (raw_value, saved_value) in enumerate(zip(ordered_rows['p_value_raw'], ordered_rows['p_value_bh'])):
        if pd.isna(raw_value):
            assert pd.isna(saved_value)
        else:
            assert abs(float(saved_value) - adjusted[index]) < 1e-12
print('BH recomputado nas cinco famílias; 27 slots preservados.')


In [ ]:
p2 = results[results['question_id'].eq('P2_DEMOGRAPHY_ENROLLMENT_DECOMPOSITION')]
group_key = p2['result_id'].str.replace(r'_(POPULATION|TERRITORIAL_RELATION)_COMPONENT$', '', regex=True)
for _, group in p2.groupby(group_key):
    total = float(group['total_enrollment_change'].iloc[0])
    components = float(group['effect_estimate'].sum())
    tolerance = float(group['closure_tolerance'].max())
    assert abs(total - components) <= tolerance
p5_reconciliation = robustness[robustness['robustness_id'].eq('P5_EPT_PANEL_BRIDGE_RECONCILIATION')]
assert len(p5_reconciliation) == 1 and float(p5_reconciliation['value'].iloc[0]) <= 1e-9
print({'p2_rows_closed': len(p2), 'p5_max_reconciliation_difference': float(p5_reconciliation['value'].iloc[0])})


## Takeaways

- **Contexto, não excepcionalidade:** o abandono de Nova Santa Rita ficou 0,95 p.p. acima da previsão fora da amostra, dentro da banda aproximada de ±3,10 p.p.
- **Demografia e organização territorial atuaram em sentidos opostos:** entre 2018 e 2025, o componente populacional foi -41,45 matrículas e o componente da relação territorial +58,45, fechando o aumento observado de 17.
- **EPT local é uma lacuna observada, não uma imputação:** o total municipal de matrículas técnicas em 2025 é zero; por isso a correspondência local é 0%, enquanto o cenário de oferta acessível no Vale alcança 46,49% dos vínculos formais do município.
- **Sinais que não devem ser promovidos como conclusão robusta:** escolaridade adulta versus EJA e escolas rurais versus matrículas mostraram magnitudes relevantes em algumas especificações, mas não passaram simultaneamente pelo BH, intervalos e estabilidade exigidos.
- **Financiamento exige cautela adicional:** o modelo principal de 2025 ajustado pela escala foi nulo; a alternativa por gasto por matrícula foi positiva, mas 2024 tinha somente 11 observações financeiras e invalidou o conjunto confirmatório.